# Очистка данных: Contacts

In [13]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
from IPython.display import display

import help_130625_dam as h

# Пути к данным
RAW_DATA_DIR = os.path.join('..', 'Sources')
PROCESSED_DATA_DIR = os.path.join('..', 'data')
REPORT_DIR = os.path.join('..', 'report')


## Загрузка и первичный осмотр

In [3]:
df = pd.read_excel(os.path.join(RAW_DATA_DIR, 'Contacts (Done).xlsx'), dtype={'Id': str})

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

print(f'Форма: {df.shape}')
df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')

n_before = df.shape[0]

h.descr_df(df, include='all', show_stats=False, show_sample_rows=True)

Форма: (18548, 4)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3
0,id,Int64,18548,0,18548,5805028000000645014,5805028000000872003,5805028000000889001
1,contact_owner_name,object,18548,0,28,Rachel White,Charlie Davis,Bob Brown
2,created_time,str,18548,0,17921,27.06.2023 11:28,03.07.2023 11:31,02.07.2023 22:37
3,modified_time,str,18548,0,16580,22.12.2023 13:34,21.05.2024 10:23,21.12.2023 13:17


После нескольких итераций и разбора источника пришел к выводу:
Пропусков нет. Дубликатов 38. Важно ИД зафиксировать как Int64. Есть один менеджер "False" у которого у которого один клиент. В сделках у этого клиента другой менеджер. Значит этот менеджер техническая ошибка. Заменяем менеджера на Jane Smith, так как она первая в звонках у этого клиента.

In [4]:
# Исправляем менеджера "False". 
mask_false = df['contact_owner_name'].astype(str).str.lower() == 'false'
if mask_false.any():
    target_id = df.loc[mask_false, 'id'].iloc[0]
    df.loc[mask_false, 'contact_owner_name'] = 'Jane Smith'
    print(f"Менеджер исправлен для контакта {target_id}")

# Обработка дублей с перепривязкой сделок и звонков
DEALS_PATH = os.path.join('..', 'data', 'cleaned', 'deals_clean.pkl')
CALLS_PATH = os.path.join('..', 'data', 'cleaned', 'calls_clean.pkl')

# Находим дубликаты по всем полям кроме ID
cols_to_check = [c for c in df.columns if c != 'id']
is_dupe = df.duplicated(subset=cols_to_check, keep=False)

if is_dupe.any():
    # Оставляем последнюю запись как "мастер-карточку"
    df_masters = df[is_dupe].sort_values('id').drop_duplicates(subset=cols_to_check, keep='last')
    
    mapping = {}
    for _, master_row in df_masters.iterrows():
        mask = True
        for col in cols_to_check:
            if pd.isna(master_row[col]):
                mask &= df[col].isna()
            else:
                mask &= (df[col] == master_row[col])
        
        all_ids = df.loc[mask, 'id'].tolist()
        m_id = master_row['id']
        
        for oid in all_ids:
            if oid != m_id:
                mapping[oid] = m_id

    if mapping:
        print(f"Сформирован маппинг для {len(mapping)} дублей.")
        
        # Сохраняем маппинг на диск — будет использован в 03 и 04
        os.makedirs(os.path.join(PROCESSED_DATA_DIR, 'cleaned'), exist_ok=True)
        pd.Series(mapping).to_pickle(os.path.join(PROCESSED_DATA_DIR, 'cleaned','contacts_mapping.pkl'))
        print(f"Маппинг сохранён: {os.path.join(PROCESSED_DATA_DIR, 'cleaned', 'contacts_mapping.pkl')}")
        
        # Перепривязываем уже существующие файлы (если есть)
        if os.path.exists(DEALS_PATH):
            deals = pd.read_pickle(DEALS_PATH)
            affected = deals['contact_id'].isin(mapping.keys()).sum()
            deals['contact_id'] = deals['contact_id'].replace(mapping)
            deals.to_pickle(DEALS_PATH)
            print(f"Deals: Перепривязано {affected} сделок к мастер-контактам.")

        if os.path.exists(CALLS_PATH):
            calls = pd.read_pickle(CALLS_PATH)
            if 'contactid' in calls.columns:
                calls['contactid'] = pd.to_numeric(calls['contactid'], errors='coerce').astype('Int64')
                affected_calls = calls['contactid'].isin(mapping.keys()).sum()
                calls['contactid'] = calls['contactid'].replace(mapping)
                calls.to_pickle(CALLS_PATH)
                print(f"Calls: Перепривязано {affected_calls} звонков.")

    # Удаляем дубликаты из основного DataFrame
    df = df.drop_duplicates(subset=cols_to_check, keep='last').reset_index(drop=True)
    print(f"В таблице Contacts {len(df)} уникальных клиентов.")
else:
    mapping = {}
    print("Бизнес-дубликатов не обнаружено.")

if target_id in mapping:
    print(f"Контакт {target_id} успешно перепривязан к мастеру {mapping[target_id]}.")

Менеджер исправлен для контакта 5805028000008772190
Сформирован маппинг для 38 дублей.
Маппинг сохранён: ../data/cleaned/contacts_mapping.pkl
Deals: Перепривязано 0 сделок к мастер-контактам.
Calls: Перепривязано 0 звонков.
В таблице Contacts 18510 уникальных клиентов.


## Типы данных: даты

In [5]:
DATE_COLS = ['created_time', 'modified_time']

for col in DATE_COLS:
    # Пробуем автоопределение формата (dayfirst=True для DD.MM.YYYY)
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT = не распарсились
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')
    
# Диапазон дат
for col in DATE_COLS:
    print(f'{col}: {df[col].min()}  →  {df[col].max()}')

Типы после парсинга:
created_time     datetime64[us]
modified_time    datetime64[us]
dtype: object

created_time: NaT = 0 (0.00%)
modified_time: NaT = 0 (0.00%)
created_time: 2023-06-27 11:28:00  →  2024-06-21 15:30:00
modified_time: 2023-07-06 10:54:00  →  2024-06-21 15:32:00


In [6]:
# Проверка логики: modified_time не должна быть раньше created_time
anomalies = df[df['modified_time'] < df['created_time']]
print(f'Строк где modified < created: {len(anomalies)}')
if len(anomalies) > 0:
    display(anomalies.head(10))

Строк где modified < created: 0


In [7]:
# Преобразование в категориальный тип для оптимизации
df['contact_owner_name'] = df['contact_owner_name'].astype('category')

owner_counts = df['contact_owner_name'].value_counts()
print(f'Уникальных менеджеров: {owner_counts.shape[0]}')

Уникальных менеджеров: 27


## Итоговый осмотр

In [8]:
print('Типы данных финального датасета:')
print(df.dtypes)
print()
h.descr_df(df, include='all', show_stats=False)

Типы данных финального датасета:
id                             Int64
contact_owner_name          category
created_time          datetime64[us]
modified_time         datetime64[us]
dtype: object



,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений
0,id,Int64,18510,0,18510
1,contact_owner_name,category,18510,0,27
2,created_time,datetime64[us],18510,0,17921
3,modified_time,datetime64[us],18510,0,16580


In [9]:
# Обогащение данных: Дата первой оплаты
# Подтягиваем информацию из buyers_info.pkl (сформирован в 03_cleaning_deals)

BUYERS_INFO_PATH = os.path.join('..', 'data', 'cleaned', 'buyers_info.pkl')

if os.path.exists(BUYERS_INFO_PATH):
    buyers_info = pd.read_pickle(BUYERS_INFO_PATH)

    # Приводим ID к типу Int64 для корректного merge
    # buyers_info['contact_id'] = pd.to_numeric(buyers_info['contact_id'], errors='coerce').astype('Int64')

    # В новом buyers_info дата первой оплаты называется became_buyer_date
    # Также там уже есть флаг is_buyer
    buyers_info = buyers_info.rename(columns={
        # 'became_buyer_date': 'first_payment_date',
        'contact_id': 'id'
    })

    # Удаляем старые столбцы если они были (защита от повторного запуска)
    df = df.drop(columns=['first_payment_date', 'is_buyer', 'new_registration_date'], errors='ignore')

    # Объединяем контакты с данными по оплатам
    df = df.merge(
        buyers_info[['id', 'first_payment_date', 'is_buyer', 'new_registration_date']],
        on='id',
        how='left'
    )

    # Заполняем пропуски для тех кто не купил
    df['is_buyer'] = df['is_buyer'].fillna(False).astype(bool)

    print(f"Контакты обогащены данными: добавлена дата первой оплаты для {df['first_payment_date'].count()} клиентов.")
    print(f"is_buyer: {df['is_buyer'].sum()} покупателей из {len(df)} контактов ({df['is_buyer'].mean()*100:.1f}%)")
else:
    df['is_buyer'] = False
    print("Файл buyers_info.pkl не найден. Сначала выполните блокнот 03_cleaning_deals.")


Контакты обогащены данными: добавлена дата первой оплаты для 814 клиентов.
is_buyer: 814 покупателей из 18510 контактов (4.4%)


## Сохранение и итоги

In [14]:
os.makedirs(os.path.join(PROCESSED_DATA_DIR, 'cleaned'), exist_ok=True)
df.to_pickle(os.path.join(PROCESSED_DATA_DIR, 'cleaned','contacts_clean.pkl'))
df.to_excel(os.path.join(PROCESSED_DATA_DIR, 'cleaned','contacts_clean.xlsx'))

# Анализ числовых полей (квартили и мода)
numeric_cols = df.select_dtypes(include=[np.number]).columns
if not numeric_cols.empty:
    print("=== Сводная статистика числовых полей ===")
    stats_list = []
    for col in numeric_cols:
        desc = df[col].describe()
        mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
        # Формируем расширенную статистику
        row = pd.Series({
            'count': desc['count'],
            'mean':  desc['mean'],
            'std':   desc['std'],
            'min':   desc['min'],
            '25%':   desc['25%'],
            '50%':   desc['50%'],
            '75%':   desc['75%'],
            'max':   desc['max'],
            'mode':  mode_val
        }, name=col)
        stats_list.append(row)
    
    stats_df = pd.DataFrame(stats_list)
    display(stats_df)

# Собираем данные для итогов
summary_data = {
    'Метрика': ['Строк исходно', 'Строк после очистки', 'Удалено дубликатов', 'Столбцы дат', 'Пропуски в менеджере'],
    'Значение': [
        n_before, 
        len(df), 
        n_before - len(df), 
        'created_time, modified_time', 
        df['contact_owner_name'].isna().sum()
    ]
}
summary_df = pd.DataFrame(summary_data)

print(f'\nСохранено в : {os.path.join(PROCESSED_DATA_DIR, "cleaned", "contacts_clean.pkl")}')
display(summary_df)


=== Сводная статистика числовых полей ===


,count,mean,std,min,25%,50%,75%,max,mode
id,18510.0,5.805028e+18,1.565980e+07,5.805028e+18,5.805028e+18,5.805028e+18,5.805028e+18,5.805028e+18,5.805028e+18



Сохранено в : ../data/cleaned/contacts_clean.pkl


,Метрика,Значение
0,Строк исходно,18548
1,Строк после очистки,18510
2,Удалено дубликатов,38
3,Столбцы дат,"created_time, modified_time"
4,Пропуски в менеджере,0


## Описательная статистика

In [11]:
# Сводная статистика для числовых и бинарных полей

def get_stats(df, col):
    stats = df[col].describe().to_frame().T
    stats['mode'] = df[col].mode()[0]
    return stats

stats_df = get_stats(df, 'is_buyer')

print("Статистика для бинарных полей (is_buyer):")
display(stats_df)

# Анализ категориальных полей
print("\nАнализ менеджеров (contact_owner_name):")
owner_stats = df['contact_owner_name'].value_counts().to_frame()
owner_stats['percentage'] = (owner_stats['count'] / len(df) * 100).round(2)
display(owner_stats.head(10))

Статистика для бинарных полей:


,count,unique,top,freq,mode,median,range
is_buyer,18510,2,False,17696,False,0.0,False - True



Анализ менеджеров (contact_owner_name):


,count,percentage
contact_owner_name,,
Charlie Davis,2018,10.90
Ulysses Adams,1809,9.77
Julia Nelson,1769,9.56
Paula Underwood,1486,8.03
Quincy Vincent,1415,7.64
Nina Scott,1148,6.20
Ben Hall,1037,5.60
Victor Barnes,966,5.22
Cara Iverson,880,4.75


## Описание датасета

**Источник:** `Contacts (Done).xlsx` — выгрузка из CRM  
**Назначение:** справочник лидов/клиентов; связывает сделки и звонки с конкретным контактом через `id`

| Столбец | Тип | Описание |
|---|---|---|
| `id` | `Int64` | Уникальный ID контакта в CRM (19-значный) |
| `contact_owner_name` | `category` | Менеджер, ответственный за контакт |
| `created_time` | `datetime` | Дата и время регистрации лида |
| `modified_time` | `datetime` | Дата последнего изменения записи |
| `first_payment_date` | `datetime` | **Обогащённый признак:** дата самого раннего платежа (из `buyers_info.pkl`); NaN = лид без оплаты |
| `is_buyer` | `bool` | **Обогащённый признак:** (из `buyers_info.pkl`); True если контакт совершил хотя бы одну оплату  (`first_payment_date` не пустая) |
| `new_registration_date` | `datetime` | **Обогащённый признак:** (из `buyers_info.pkl`); Дата сделки в случае когда первая сделка раньше регистрации контакта |

**Объём:** ~18 510 контактов, 7 столбцов после обогащения  
**Ключевые связи:**
- `id` → `deals.contact_id` (сделки)
- `id` → `calls.contactid` (звонки)

## Выводы

В исходных данных CRM обнаружено **38 дублирующихся контактов** и технически некорректный менеджер `"False"` (1 запись). Дубли появляются при повторном создании контакта в CRM — например, когда клиент обращается повторно и менеджер регистрирует новую карточку вместо поиска существующей. Менеджер `"False"` — результат программного сбоя при импорте данных.

Без очистки каждый такой контакт учитывался бы как отдельный лид: метрика конверсии (Leads → Buyers) была бы искусственно занижена, а связанные сделки и звонки не объединялись бы на одного клиента — корректный расчёт LTV и анализ воронки были бы невозможны.

**Что сделано:** дубли удалены с перепривязкой сделок и звонков к мастер-ID через `contacts_mapping.pkl`; менеджер исправлен с `"False"` на `Jane Smith` (первый контакт по звонкам). Датасет обогащён флагом `is_buyer` и датой первого платежа `first_payment_date` из `buyers_info.pkl`.

> **Системная рекомендация:** настроить в CRM валидацию на дублирование контактов по номеру телефона или email ещё на этапе ввода данных. Менеджеров обязать при обнаружении дубля сразу же принимать меры к объединению контактов.